In [1]:
!python --version

Python 3.14.3


The system cannot find the path specified.


# ___interactive Tree Of Life (iTOL)___
-----------------

In [2]:
# https://itol.embl.de/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps

In [5]:
# it's critical that the dataset is name matched with the phylogeny
data = pd.read_csv(r"./../../data/chapter2/FREDv3subset/collab_fineroots_log_995_species_means_5states_name_matched_with_phylogeny.csv")
data

,binominal,F01286,F01287,F01289,F01290,F00056,F00004,F00679,F00727,state
0,Rudbeckia_hirta,Rudbeckia,hirta,Asteraceae,Asterales,NaN,"Poon GT, Maherali H. 2015. Competitive interac...",-1.332792,5.694268,AM
1,Ratibida_pinnata,Ratibida,pinnata,Asteraceae,Asterales,NaN,"Craine JM, Froehle J, Tilman DG, Wedin DA, Cha...",-0.820981,4.025352,AM
2,Heliopsis_helianthoides,Heliopsis,helianthoides,Asteraceae,Asterales,NaN,"Craine JM, Froehle J, Tilman DG, Wedin DA, Cha...",-0.967584,4.158883,NM
3,Liatris_aspera,Liatris,aspera,Asteraceae,Asterales,NaN,"Craine JM, Froehle J, Tilman DG, Wedin DA, Cha...",-0.798508,4.382027,AM
4,Arnica_sororia,Arnica,sororia,Asteraceae,Asterales,NaN,"Kembel SW, Cahill JF, Jr. 2011. Independent Ev...",-1.413460,4.342257,AM
...,...,...,...,...,...,...,...,...,...,...
990,Dicranopteris_linearis,Dicranopteris,linearis,Gleicheniaceae,Gleicheniales,1.0,"Kong DL, Wang JJ, Kardol P, Wu HF, Zeng H, Den...",-1.625567,4.318821,AM
991,Osmunda_japonica,Osmunda,japonica,Osmundaceae,Osmundales,1.0,"Wang R, Wang Q, Zhao N, Xu Z, Zhu X, Jiao C, Y...",-0.870421,3.157944,AM
992,Equisetum_hyemale,Equisetum,hyemale,Equisetaceae,Equisetales,1.0,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",-0.736703,4.454201,NMAM
993,Equisetum_pratense,Equisetum,pratense,Equisetaceae,Equisetales,1.0,"Dong X, Wang H, Gu J, Wang Y, Wang Z. 2014. Ro...",-1.413391,5.540389,NMAM


In [6]:
# tip lebels - iTOL leaf IDs - to create coloured ranges
order_ranges = data.groupby("F01290").agg({"binominal": lambda s: (s.values[0], s.values[-1])}) # capture the first and the last species in every order (species sequence matched with phylogeny)
order_ranges

,binominal
F01290,
Apiales,"(Musineon_divaricatum, Pennantia_corymbosa)"
Aquifoliales,"(Ilex_macropoda, Ilex_pedunculosa)"
Arecales,"(Rhopalostylis_sapida, Calamus_rhabdocladus)"
Asparagales,"(Convallaria_majalis, Astelia_nervosa)"
Asterales,"(Rudbeckia_hirta, Carpodetus_serratus)"
Brassicales,"(Alliaria_petiolata, Descurainia_sophia)"
Canellales,"(Pseudowintera_colorata, Pseudowintera_colorata)"
Caryophyllales,"(Silene_rupestris, Armeria_velutina)"
Celastrales,"(Prionostemma_aspera, Microtropis_discolor)"


In [7]:
np.unique(np.random.randint(low=0, high=255, size=(100, 3)), axis=1)[:4]

array([[129, 144, 217],
       [  0, 171, 249],
       [183, 194, 243],
       [202,  61, 221]], dtype=int32)

In [8]:
# looks like leaf id's need to be the binominal names???
boilerplate = "TREE_COLORS\nSEPARATOR SPACE\nDATA\n"  # boilerplate needed for TREE_COLORS type dataset
colours = np.unique(np.random.randint(low=0, high=255, size=(500, 3)), axis=1)[:order_ranges.shape[0]] # get unique colours for each order in the dataset

with open(file=r"../../data/chapter2/iTreeOfLife/TREE_COLORS.txt", mode="wt") as fp:
    fp.write(boilerplate)
    for (r, g, b), (first, last) in zip(colours, order_ranges.binominal):
            fp.write(f"{first}|{last} range rgba({r},{g},{b},0.5)\n")

In [11]:
data.state.unique()

<StringArray>
['AM', 'NM', 'NMAM', 'EcMAM', 'EcM']
Length: 5, dtype: str

In [31]:
STATE_COLOURS = {"AM": "#FF0000", "NM": "#00FF00", "NMAM": "#0000FF", "EcMAM": "#FFFF00", "EcM": "#00FFFF"}
STATE_COLOURS

{'AM': '#FF0000',
 'NM': '#00FF00',
 'NMAM': '#0000FF',
 'EcMAM': '#FFFF00',
 'EcM': '#00FFFF'}

In [34]:
# lets use branch symbol colours to differentiate mycorrhizal states
boilerplate = "DATASET_SYMBOL\nSEPARATOR COMMA\nDATASET_LABEL,MYCORRHIZAL_STATES\nDATA\n"

# the following fields are required for each node: ID,symbol,size,color,fill,position,label
SYMBOL = 2 # 2 is circles
SYMBOL_SIZE = 5
FILL_INSIDE = 1 # 1 will fill the shape; 0 will only colour the outline of the shape
BRANCH_POS = 1 # we want the symbols to be located at the tips (end of the branches)

with open(file=r"../../data/chapter2/iTreeOfLife/DATASET_SYMBOL.txt", mode="wt") as fp:
    fp.write(boilerplate)
    for (_, (name, state)) in data.loc[:, ["binominal", "state"]].iterrows():
        fp.write(f"{name},{SYMBOL},{SYMBOL_SIZE},{STATE_COLOURS.get(state)},{FILL_INSIDE},{BRANCH_POS},{state}\n")

In [ ]:
"""
#LEGEND_TITLE,Dataset legend
#LEGEND_SCALE,1
#LEGEND_POSITION_X,100
#LEGEND_POSITION_Y,100
#LEGEND_HORIZONTAL,0
#LEGEND_VISIBLE,1
#LEGEND_SHAPES,1,2,3
#LEGEND_COLORS,#ff0000,#00ff00,#0000ff
#LEGEND_LABELS,value1,value2,value3
#LEGEND_SHAPE_SCALES,1,1,0.5
#LEGEND_SHAPE_INVERT,0,0,0
"""